# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring a Croissant schema-based dataset using the `mlcroissant` library. We will:
- Load the metadata and records,
- Give an overview of record sets and fields,
- Extract data for analysis,
- Perform exploratory data analysis (EDA), and
- Visualize and summarize insights.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset title: {metadata.name}\n")
print(f"Description: {metadata.description}\n")

## 2. Data Overview

Review available record sets, fields, and their `@id` identifiers. This step is crucial to understanding what data is available and how to address it for further processing. All references will use the Croissant `@id`.

In [ ]:
# List all available record sets and their fields (using @id).
print('Available record sets:')
record_sets = []
for record_set in dataset.metadata.record_sets:
    print(f"  Record set: @id={record_set.id}, name={record_set.name}")
    record_sets.append(record_set.id)
    if hasattr(record_set, 'fields') and record_set.fields:
        for field in record_set.fields:
            print(f"    Field: @id={field.id}, name={field.name}, dataType={field.data_type}")

## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s discovered above. This approach can be extended to all record sets in the dataset.

In [ ]:
# Extract data from all record sets into dataframes
dataframes = {}
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    # Some record sets may not have records. Skip if empty.
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records from record set: {record_set_id}")

# Display one example record set with its column names and head
if len(dataframes) > 0:
    # Use the first available record set with data
    example_record_set_id = list(dataframes.keys())[0]
    print(f"\nColumns for record set '@id': {example_record_set_id}")
    print(dataframes[example_record_set_id].columns.tolist())
    display(dataframes[example_record_set_id].head())
else:
    print('No data was found in any record set.')

## 4. Exploratory Data Analysis (EDA)

We demonstrate some common practices for EDA using one record set. We will:
- Filter rows on a chosen numeric field,
- Normalize the values,
- Group by another field if available.

**Note**: Replace the field `@id` below as appropriate according to your dataset's overview above.

In [ ]:
# Select a record set and numeric field for analysis.
# You may replace the following IDs with those fitting your analysis, as per the overview output above.
record_set_id = example_record_set_id  # Use the same as shown previously
df = dataframes[record_set_id]

# Replace with an actual numeric field @id found previously
# (For demonstration, pick the first numeric column if possible)
numeric_field = None
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field = col
        break
if numeric_field is None:
    print('No numeric field found in the selected record set. Please check your dataset overview!')
else:
    threshold = df[numeric_field].mean() if not pd.isnull(df[numeric_field].mean()) else 0
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize numeric field
    filtered_df[numeric_field + '_normalized'] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, numeric_field + '_normalized']].head())

    # Try grouping by another field (if any categorical field exists)
    group_field = None
    for col in df.columns:
        if col != numeric_field and df[col].nunique() < len(df) // 2:
            group_field = col
            break
    if group_field is not None:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"\nGrouped data by '{group_field}':")
        display(grouped_df.head())
    else:
        print('No suitable categorical field for grouping found.')

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset. Below we use simple matplotlib/Pandas visualizations. You can extend with seaborn or plotly as needed.

In [ ]:
import matplotlib.pyplot as plt

if numeric_field is not None:
    fig, ax = plt.subplots(figsize=(8,4))
    df[numeric_field].hist(ax=ax, bins=30)
    ax.set_title(f'Histogram of {numeric_field}')
    ax.set_xlabel(numeric_field)
    ax.set_ylabel('Frequency')
    plt.show()

    if group_field is not None and group_field in df.columns:
        # Boxplot by group field
        plt.figure(figsize=(10, 5))
        df[[group_field, numeric_field]].boxplot(by=group_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.suptitle('')
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()

## 6. Conclusion

This notebook demonstrates how to explore and analyze a Croissant-based dataset using `mlcroissant`. By referring to entity `@id`s for record sets and fields, workflows are robust and interoperable. You can further tune analysis and visualization steps to your own needs according to this dataset’s structure.

Key points:
- Used `mlcroissant` to easily access metadata and records defined by a Croissant schema
- Explored dataset structure via `@id` references
- Demonstrated filtering, normalization, and grouping, plus a sample visualization workflow

Extend this notebook for your research, and consult the documentation for `mlcroissant` and Croissant specifications for advanced scenarios.